# 02 Jacobian And Network Control


In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp


def sigmoid(x, a, theta):
    z = np.clip(-a * (x - theta), -500, 500)
    return 1.0 / (1.0 + np.exp(z)) - 1.0 / (1.0 + np.exp(a * theta))


def sigmoid_prime(x, a, theta):
    z = np.clip(-a * (x - theta), -500, 500)
    logistic = 1.0 / (1.0 + np.exp(z))
    return a * logistic * (1.0 - logistic)


def wc_jacobian(E_star, I_star, W, params):
    N = W.shape[0]
    eye = np.eye(N)

    tau = params["tau"]
    c1, c2, c3 = params["c1"], params["c2"], params["c3"]
    c4, c5, c6 = params["c4"], params["c5"], params["c6"]
    r_e, r_i = params["r_e"], params["r_i"]
    aE, thetaE = params["aE"], params["thetaE"]
    aI, thetaI = params["aI"], params["thetaI"]

    Smax_e = 1.0 - 1.0 / (1.0 + np.exp(aE * thetaE))
    Smax_i = 1.0 - 1.0 / (1.0 + np.exp(aI * thetaI))

    hE = c1 * E_star - c2 * I_star + c5 * (W @ E_star)
    hI = c3 * E_star - c4 * I_star + c6 * (W @ I_star)

    S_E = sigmoid(hE, aE, thetaE)
    S_I = sigmoid(hI, aI, thetaI)

    D_E = np.diag((2.0 * Smax_e - r_e * E_star) * sigmoid_prime(hE, aE, thetaE))
    D_I = np.diag((2.0 * Smax_i - r_i * I_star) * sigmoid_prime(hI, aI, thetaI))

    A_EE = (-eye - r_e * np.diag(S_E) + D_E @ (c1 * eye + c5 * W)) / tau
    A_EI = (D_E @ (-c2 * eye)) / tau
    A_IE = (D_I @ (c3 * eye)) / tau
    A_II = (-eye - r_i * np.diag(S_I) + D_I @ (-c4 * eye + c6 * W)) / tau
    return np.block([[A_EE, A_EI], [A_IE, A_II]])


def dominant_target_mode(A):
    eigvals, eigvecs = np.linalg.eig(A)
    idx = int(np.argmax(np.real(eigvals)))
    target = np.real(eigvecs[:, idx])
    if np.linalg.norm(target) < 1e-12:
        target = np.imag(eigvecs[:, idx])
    return target / np.linalg.norm(target), eigvals[idx]


def minimum_control_energy(A, target, control_coordinate, T=1.0, nctpy_dt=0.001):
    from nctpy.energies import get_control_inputs, integrate_u

    n_state = A.shape[0]
    B = np.zeros((n_state, n_state))
    B[control_coordinate, control_coordinate] = 1.0

    x, u, n_err = get_control_inputs(
        A_norm=A,
        T=T,
        B=B,
        x0=np.zeros((n_state, 1)),
        xf=target.reshape(-1, 1),
        system="continuous",
        xr="zero",
        S=np.zeros((n_state, n_state)),
        expm_version="eig",
    )
    return float(np.sum(integrate_u(u)) * nctpy_dt)


def rank_minimum_energy_levers(A, node_names, control_population="E"):
    target, leading_eigenvalue = dominant_target_mode(A)
    N = len(node_names)
    offset = 0 if control_population == "E" else N

    rows = []
    for region_index, node_name in enumerate(node_names):
        rows.append({
            "node_name": node_name,
            "control_coordinate": offset + region_index,
            "energy": minimum_control_energy(A, target, offset + region_index),
            "leading_eigenvalue_real": float(np.real(leading_eigenvalue)),
        })

    df = pd.DataFrame(rows).sort_values("energy").reset_index(drop=True)
    df["rank"] = np.arange(1, len(df) + 1)
    return df


# Pipeline shape:
# 1. simulate Wilson-Cowan at c5_pre to get E_star and I_star
# 2. A = wc_jacobian(E_star, I_star, W, params_at_c5_pre)
# 3. rankings = rank_minimum_energy_levers(A, node_names, control_population="E")


## Ranking CSV Output


**CSV:** `data/network_control/105115_scale33_network_control_rankings.csv`

| rank | node_name | node | control_coordinate | control_type | energy | c5_operating | c5_star | lambda_max_real | binary_degree | weighted_degree | mean_connection_weight | max_connection_weight |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1.0 | rh.insula | 34 | 33 | E | 6306087302.263114 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 30 | 1.022984516900698 | 0.03409948389668993 | 0.18023911372085932 |
| 2.0 | rh.precentral | 10 | 9 | E | 77236737711.6637 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 1.5588025784185224 | 0.07422869421040583 | 0.36645972010919786 |
| 3.0 | lh.supramarginal | 58 | 57 | E | 346626721125.8311 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 31 | 1.0227659418566992 | 0.03299244973731288 | 0.24515590178850122 |
| 4.0 | rh.inferiorparietal | 19 | 18 | E | 619021343084.9258 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 30 | 1.6678342076834822 | 0.055594473589449404 | 0.3510368517606976 |
| 5.0 | rh.paracentral | 11 | 10 | E | 16350607902504.377 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 17 | 0.5247453696547637 | 0.030867374685574336 | 0.20750235213573331 |
| 6.0 | lh.posteriorcingulate | 55 | 54 | E | 21545272975653.094 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 36 | 0.9307618416291092 | 0.02585449560080859 | 0.2742956869225843 |
| 7.0 | lh.bankssts | 72 | 71 | E | 48288883242271.68 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 0.4668496384882538 | 0.022230935166107323 | 0.12467307265745856 |
| 8.0 | lh.precentral | 51 | 50 | E | 59380457301479.86 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 19 | 1.3522171752478573 | 0.07116932501304513 | 0.24637672337571398 |
| 9.0 | lh.postcentral | 57 | 56 | E | 63302025238551.59 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 18 | 0.8880384171752961 | 0.04933546762084978 | 0.24515590178850122 |
| 10.0 | Right-Putamen | 37 | 36 | E | 76547103631510.77 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 30 | 1.285269238600617 | 0.0428423079533539 | 0.21332391184418903 |
| 11.0 | rh.posteriorcingulate | 14 | 13 | E | 88219140418053.78 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 29 | 0.8662342237617578 | 0.029870145646957166 | 0.23674875924249883 |
| 12.0 | rh.caudalmiddlefrontal | 9 | 8 | E | 101793792907815.95 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 1.2245586873552932 | 0.058312318445490154 | 0.36645972010919786 |
| 13.0 | rh.lateraloccipital | 23 | 22 | E | 126576231737341.9 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 17 | 1.130710027000299 | 0.06651235452942936 | 0.43289587128756307 |
| 14.0 | Right-Thalamus-Proper | 35 | 34 | E | 129894244408825.31 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 42 | 1.14033266003488 | 0.027150777619878094 | 0.1974905489008617 |
| 15.0 | lh.lateraloccipital | 64 | 63 | E | 282706143474243.1 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 15 | 0.604503936319787 | 0.040300262421319136 | 0.29228281371409487 |
| 16.0 | rh.bankssts | 31 | 30 | E | 309963163746784.7 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 20 | 0.24541179452293882 | 0.012270589726146941 | 0.09880125098608909 |
| 17.0 | lh.superiorfrontal | 49 | 48 | E | 485019330359128.6 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 19 | 0.8089089201490984 | 0.04257415369205781 | 0.23178550641413595 |
| 18.0 | lh.superiorparietal | 59 | 58 | E | 487069739895730.2 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 31 | 1.4410439406895137 | 0.04648528840933915 | 0.38709107182323044 |
| 19.0 | rh.supramarginal | 17 | 16 | E | 636771557554335.8 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 29 | 0.9065959715206572 | 0.03126193005243645 | 0.22753128970411066 |
| 20.0 | rh.parstriangularis | 5 | 4 | E | 751599826727682.6 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 13 | 0.04907276292704635 | 0.0037748279174651036 | 0.01928258375960094 |
| 21.0 | rh.rostralmiddlefrontal | 7 | 6 | E | 818571456801273.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 23 | 0.7773914650241991 | 0.03379962891409561 | 0.2636921317393258 |
| 22.0 | rh.postcentral | 16 | 15 | E | 847200596755720.2 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 19 | 0.7886400831421858 | 0.04150737279695715 | 0.18722818403018662 |
| 23.0 | rh.pericalcarine | 22 | 21 | E | 892453922009022.5 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 12 | 0.6984752119433663 | 0.05820626766194719 | 0.43289587128756307 |
| 24.0 | lh.inferiorparietal | 60 | 59 | E | 928930830558485.8 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 25 | 1.1432274465932053 | 0.04572909786372821 | 0.2749993919422878 |
| 25.0 | lh.paracentral | 52 | 51 | E | 933234501534573.1 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 17 | 0.6144517663710493 | 0.03614422155123819 | 0.24637672337571398 |
| 26.0 | Right-Caudate | 36 | 35 | E | 1019054474308842.8 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 36 | 1.1088951713894923 | 0.03080264364970812 | 0.23674875924249883 |
| 27.0 | Brain-Stem | 83 | 82 | E | 1743119334468332.8 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 10 | 0.04760137970403008 | 0.004760137970403008 | 0.01768858526799998 |
| 28.0 | Right-Amygdala | 41 | 40 | E | 1874519415456221.2 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 8 | 0.013738241180119333 | 0.0017172801475149166 | 0.008423135841904753 |
| 29.0 | lh.precuneus | 61 | 60 | E | 1884993561096289.2 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 28 | 1.3497168899884564 | 0.04820417464244487 | 0.38709107182323044 |
| 30.0 | lh.caudalmiddlefrontal | 50 | 49 | E | 1949540175813459.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 1.149694069236389 | 0.05474733663030424 | 0.2422504530329075 |
| 31.0 | lh.insula | 75 | 74 | E | 2004023905250734.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 27 | 0.9123695513414058 | 0.033791464864496516 | 0.24024062884784542 |
| 32.0 | rh.precuneus | 20 | 19 | E | 2283851379612504.5 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 31 | 1.1854550788740454 | 0.03824048641529179 | 0.3510368517606976 |
| 33.0 | rh.lingual | 24 | 23 | E | 4383641804986226.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 0.6350191449014725 | 0.030239006900070118 | 0.2883697873166531 |
| 34.0 | rh.superiorparietal | 18 | 17 | E | 4999458962745336.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 30 | 0.885804686847601 | 0.029526822894920035 | 0.271315602786113 |
| 35.0 | Left-Thalamus-Proper | 76 | 75 | E | 5176447400344439.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 48 | 1.4211056317979165 | 0.02960636732912326 | 0.2742956869225843 |
| 36.0 | rh.middletemporal | 30 | 29 | E | 5272942520728058.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 19 | 0.16538667292620957 | 0.008704561732958398 | 0.056914809017832364 |
| 37.0 | rh.inferiortemporal | 29 | 28 | E | 7019348543144716.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 16 | 0.1539994462437358 | 0.009624965390233487 | 0.05167967015912954 |
| 38.0 | Left-Caudate | 77 | 76 | E | 8355812054139664.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 36 | 1.0020706169590572 | 0.027835294915529368 | 0.14698905153987205 |
| 39.0 | lh.middletemporal | 71 | 70 | E | 9324820767823982.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 17 | 0.0882829933809764 | 0.005193117257704494 | 0.02158028727090534 |
| 40.0 | lh.caudalanteriorcingulate | 54 | 53 | E | 9837087281809518.0 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 26 | 0.405216807179228 | 0.015585261814585692 | 0.1360549682412982 |
| 41.0 | rh.parsopercularis | 6 | 5 | E | 1.259048672731721e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 16 | 0.322286236826905 | 0.020142889801681562 | 0.10501198089483531 |
| 42.0 | rh.fusiform | 25 | 24 | E | 1.272559638856729e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 16 | 0.4087246700804769 | 0.025545291880029807 | 0.20361598123146207 |
| 43.0 | Left-Putamen | 78 | 77 | E | 1.308966385696265e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 26 | 0.8578217501171213 | 0.03299314423527389 | 0.17754157781199617 |
| 44.0 | Right-Pallidum | 38 | 37 | E | 1.9384161375614332e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 26 | 0.2209100652005374 | 0.008496540969251438 | 0.04595407022608795 |
| 45.0 | rh.isthmuscingulate | 15 | 14 | E | 2.014497813812786e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 25 | 0.484047762681915 | 0.019361910507276602 | 0.165269388756259 |
| 46.0 | lh.pericalcarine | 63 | 62 | E | 4.991950775318215e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 13 | 0.7263728510957002 | 0.055874834699669244 | 0.29228281371409487 |
| 47.0 | Left-Hippocampus | 81 | 80 | E | 5.090780808061425e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 29 | 0.27673199899837575 | 0.009542482724081923 | 0.06700124763358159 |
| 48.0 | Right-Hippocampus | 40 | 39 | E | 5.134637705046519e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 24 | 0.1825581416267002 | 0.007606589234445842 | 0.04382696187107529 |
| 49.0 | lh.rostralmiddlefrontal | 48 | 47 | E | 6.890874325039358e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 19 | 0.5207523767778102 | 0.02740801983041106 | 0.11005520020270992 |
| 50.0 | rh.caudalanteriorcingulate | 13 | 12 | E | 7.078633195006182e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 0.2284855563596176 | 0.010880264588553219 | 0.09553861862200953 |
| 51.0 | lh.isthmuscingulate | 56 | 55 | E | 7.309951360183358e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 29 | 0.6231841059337327 | 0.0214891071011632 | 0.15578536428616496 |
| 52.0 | lh.cuneus | 62 | 61 | E | 8.668002080638765e+16 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 13 | 0.3046616247425144 | 0.02343550959557803 | 0.12302576317951643 |
| 53.0 | Left-Pallidum | 79 | 78 | E | 1.236780097650709e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 21 | 0.1495266544897117 | 0.007120316880462461 | 0.052111489148493007 |
| 54.0 | lh.superiortemporal | 73 | 72 | E | 1.7299847853574474e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 20 | 0.21789799447226135 | 0.010894899723613068 | 0.081427200537502 |
| 55.0 | lh.inferiortemporal | 70 | 69 | E | 2.091233746885782e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 19 | 0.20133427101606002 | 0.010596540579792632 | 0.058503476410799206 |
| 56.0 | lh.parsopercularis | 47 | 46 | E | 2.219016957172801e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 14 | 0.16985413358159956 | 0.012132438112971397 | 0.042222301182206094 |
| 57.0 | lh.lingual | 65 | 64 | E | 2.4485715516344464e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 20 | 0.5925142954916581 | 0.029625714774582906 | 0.15719810542420595 |
| 58.0 | lh.parstriangularis | 46 | 45 | E | 3.304208723516687e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 10 | 0.03009405178959008 | 0.003009405178959008 | 0.02201743735890293 |
| 59.0 | rh.superiortemporal | 32 | 31 | E | 5.110361864338355e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 12 | 0.019378543535015048 | 0.0016148786279179207 | 0.0058855328920650925 |
| 60.0 | rh.cuneus | 21 | 20 | E | 8.89347268184317e+17 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 9 | 0.14249493539131147 | 0.01583277059903461 | 0.05916453264142971 |
| 61.0 | lh.transversetemporal | 74 | 73 | E | 1.200467976874257e+18 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 16 | 0.02425649878523204 | 0.0015160311740770025 | 0.006658542194012048 |
| 62.0 | rh.transversetemporal | 33 | 32 | E | 1.2053784653564168e+18 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 9 | 0.008700352970878834 | 0.0009667058856532038 | 0.005149841280556957 |
| 63.0 | Right-Accumbens-area | 39 | 38 | E | 1.7036817905680888e+18 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 2 | 0.0012101593899445434 | 0.0006050796949722717 | 0.0009222800637022291 |
| 64.0 | lh.parahippocampal | 67 | 66 | E | 5.079536278713634e+18 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 10 | 0.010374317941991548 | 0.001037431794199155 | 0.0029267731501301957 |
| 65.0 | rh.parahippocampal | 26 | 25 | E | 9.659162632178035e+18 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 10 | 0.008972239001218798 | 0.0008972239001218798 | 0.0015566808011621442 |
| 66.0 | lh.fusiform | 66 | 65 | E | 1.4414242498230585e+19 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 20 | 0.2432633617733897 | 0.012163168088669485 | 0.049115411716119294 |
| 67.0 | rh.superiorfrontal | 8 | 7 | E | 1.766082445254676e+19 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 22 | 0.6854779934733892 | 0.031158090612426782 | 0.2636921317393258 |
| 68.0 | lh.lateralorbitofrontal | 42 | 41 | E | 5.045662434878481e+19 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 1 | 0.000549103159314044 | 0.000549103159314044 | 0.000549103159314044 |
| 69.0 | rh.rostralanteriorcingulate | 12 | 11 | E | 5.359557447497582e+19 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 3 | 0.0005970830470210964 | 0.00019902768234036547 | 0.0003038726221446651 |
| 70.0 | Left-Amygdala | 82 | 81 | E | 1.9121128681717826e+20 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 5 | 0.0013594301516998177 | 0.00027188603033996355 | 0.0007943336964834227 |
| 71.0 | rh.lateralorbitofrontal | 1 | 0 | E | 8.084094184687674e+20 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 6 | 0.015055022542746214 | 0.0025091704237910355 | 0.010416966731064485 |
| 72.0 | lh.rostralanteriorcingulate | 53 | 52 | E | 1.1637176415385224e+21 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 5 | 0.006525264728159124 | 0.001305052945631825 | 0.003438558619005421 |
| 73.0 | lh.entorhinal | 68 | 67 | E | 7.426659651174933e+30 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 74.0 | lh.medialorbitofrontal | 45 | 44 | E | 1.591385232599936e+32 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 75.0 | Left-Accumbens-area | 80 | 79 | E | 1.6446294239188503e+32 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 76.0 | lh.temporalpole | 69 | 68 | E | 4.6368421244266675e+32 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 77.0 | rh.parsorbitalis | 2 | 1 | E | 3.134510592099286e+33 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 78.0 | rh.medialorbitofrontal | 4 | 3 | E | 6.624658967560905e+33 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 79.0 | rh.temporalpole | 28 | 27 | E | 6.817390358435295e+33 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 80.0 | lh.frontalpole | 44 | 43 | E | 9.721966580400232e+33 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 81.0 | rh.entorhinal | 27 | 26 | E | 2.604401042946816e+34 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 82.0 | lh.parsorbitalis | 43 | 42 | E | 8.892801979167743e+34 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
| 83.0 | rh.frontalpole | 3 | 2 | E | 3.983987746547828e+35 | 9.200000000000001 | 9.4 | -0.08284449781747101 | 0 | 0.0 | 0.0 | 0.0 |
